# Module 5 Live-Coding Demo — Embeddings & Vector DBs from Scratch
## The NaijaThreads Story

**Generative AI Fellowship — Beginner | Week 10 Module Demo**

**NaijaThreads** is a small Lagos-based online fashion store. Right now, if a customer types a question into their FAQ search box that doesn't use the *exact* words on the page, they get nothing back — and the founder is losing sales to confused, frustrated customers.

Over this notebook, we'll take NaijaThreads from **raw text with no understanding of meaning at all**, to a **working semantic search engine** — following the same arc as this week's five topics, but as one continuous story.

**The journey:**
1. **Turn words into vectors** — train a small embedding model on NaijaThreads' own text (Topics 1–2)
2. **Measure similarity by hand** — teach ourselves exactly what "closeness" means, in code (Topic 3)
3. **Move from brute force to a real vector database** — see why NaijaThreads can't just compare-against-everything forever (Topics 3–4)
4. **Assemble the full semantic search engine** — and prove it beats keyword search on a real customer question (Topic 5)

Let's get started.

## Setup

In [ ]:
!pip install chromadb gensim numpy --quiet

In [ ]:
import numpy as np
import time
import chromadb
from gensim.models import Word2Vec
from pprint import pprint

print("Ready to go!")

---
## Part 1 — Turning Words Into Vectors

NaijaThreads has 20 FAQ entries — their entire searchable knowledge. On their own, these are just strings of characters to a computer. Our first job: teach a model to represent them as vectors that capture meaning.

In [ ]:
faqs = [
    {"id": "faq1",  "category": "delivery", "text": "orders within lagos are delivered within 2 to 3 business days"},
    {"id": "faq2",  "category": "delivery", "text": "orders outside lagos take 5 to 7 business days to arrive"},
    {"id": "faq3",  "category": "returns",  "text": "you can return an item within 7 days of delivery if unused and in original packaging"},
    {"id": "faq4",  "category": "returns",  "text": "refunds are processed within 5 business days after we receive your returned item"},
    {"id": "faq5",  "category": "product",  "text": "check our size chart on each product page before ordering"},
    {"id": "faq6",  "category": "returns",  "text": "we allow one free size exchange within 7 days of delivery"},
    {"id": "faq7",  "category": "payment",  "text": "we accept bank transfer debit card and cash on delivery"},
    {"id": "faq8",  "category": "payment",  "text": "cash on delivery is only available within lagos"},
    {"id": "faq9",  "category": "orders",   "text": "you will receive a tracking link via sms once your order ships"},
    {"id": "faq10", "category": "orders",   "text": "orders can be cancelled within 1 hour of placing them"},
    {"id": "faq11", "category": "payment",  "text": "enter your discount code at checkout to apply a reduction"},
    {"id": "faq12", "category": "product",  "text": "if an item is out of stock you can join the waitlist to be notified"},
    {"id": "faq13", "category": "support",  "text": "reach our support team via whatsapp or email for help"},
    {"id": "faq14", "category": "orders",   "text": "for bulk or wholesale orders please contact us directly"},
    {"id": "faq15", "category": "delivery", "text": "we currently do not ship outside nigeria"},
    {"id": "faq16", "category": "orders",   "text": "gift wrapping is available for an additional fee at checkout"},
    {"id": "faq17", "category": "product",  "text": "hand wash our ankara items in cold water to preserve the print"},
    {"id": "faq18", "category": "support",  "text": "we do not have a physical store all orders are online only"},
    {"id": "faq19", "category": "orders",   "text": "earn points on every purchase through our loyalty program"},
    {"id": "faq20", "category": "orders",   "text": "contact support immediately to change your delivery address before shipping"},
]

print(f"NaijaThreads FAQ page: {len(faqs)} entries")
pprint(faqs[6])

20 FAQs alone is too small a corpus to teach a model that everyday words like "pay" or "package" relate to words NaijaThreads actually uses, like "transfer" or "delivered." So — just like Topic 5 — we add background sentences purely to broaden what the model learns. These will **never** be shown to a customer; they only shape the embedding space.

In [ ]:
background_sentences = [
    "my package will arrive within a few business days after it ships",
    "the courier delivers parcels within lagos every business day",
    "track your parcel until it arrives at your address",
    "your order arrives once it has been delivered to your address",
    "a delivered order usually arrives within a few business days",
    "customers can return a parcel if the item does not fit",
    "exchange your order for a different size or color if needed",
    "refund requests are reviewed after the returned item arrives",
    "pay with a debit card or transfer when you checkout",
    "you can select your payment method at checkout before you pay",
    "contact an agent through whatsapp if you need help",
    "apply your discount code before you complete checkout",
    "join the waitlist if the item you want is out of stock",
]

training_sentences = [faq["text"].split() for faq in faqs] + [s.split() for s in background_sentences]

word_model = Word2Vec(
    sentences=training_sentences,
    vector_size=25,
    window=5,
    min_count=1,
    sg=1,
    epochs=400,
    seed=42,
    workers=1,
)

print("Vocabulary size:", len(word_model.wv.key_to_index))

Let's sanity-check what the model learned, the same way we did in Topic 2 — find the nearest neighbours of a word.

In [ ]:
pprint(word_model.wv.most_similar("arrive", topn=5))

`package` comes out as the single closest word — a genuine win, since "package" never appears anywhere near "arrive" by coincidence; the model learned the connection purely from the background sentences we added. You'll also spot a common word or two further down the list (a familiar "small corpus" side effect from Topics 2 and 3) — the meaningful signal is still clearly there at the top.

---
## Part 2 — Measuring Similarity, By Hand

Before we let any tool do the searching for us, let's prove we understand *exactly* what "similar" means — by writing the formulas ourselves, same as Topic 3.

In [ ]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def euclidean_distance(a, b):
    return np.linalg.norm(a - b)


STOPWORDS = {
    "i", "a", "an", "the", "is", "are", "was", "were", "will", "when", "my", "your", "our",
    "you", "we", "to", "of", "in", "on", "for", "at", "and", "or", "if", "it", "its", "this",
    "that", "do", "does", "did", "can", "not", "no", "once", "after", "before", "within", "via", "with",
}


def sentence_vector(sentence, model):
    words = [w for w in sentence.split() if w not in STOPWORDS]
    word_vectors = [model.wv[w] for w in words if w in model.wv]
    if not word_vectors:
        word_vectors = [model.wv[w] for w in sentence.split() if w in model.wv]
    return np.mean(word_vectors, axis=0)


faq_embeddings = [sentence_vector(faq["text"], word_model) for faq in faqs]
print("FAQ embeddings ready:", len(faq_embeddings))

In [ ]:
vec_delivery = sentence_vector(faqs[0]["text"], word_model)   # delivery FAQ
vec_returns  = sentence_vector(faqs[2]["text"], word_model)   # returns FAQ
vec_payment  = sentence_vector(faqs[6]["text"], word_model)   # payment FAQ

print("cosine(delivery, returns) =", cosine_similarity(vec_delivery, vec_returns))
print("cosine(delivery, payment) =", cosine_similarity(vec_delivery, vec_payment))
print()
print("euclidean(delivery, returns) =", euclidean_distance(vec_delivery, vec_returns))
print("euclidean(delivery, payment) =", euclidean_distance(vec_delivery, vec_payment))

Two different metrics, computed from the same formulas we wrote in Topic 3 — this is the exact math running underneath any search tool we use from here on. Nothing about a vector database is magic; it's this.

---
## Part 3 — From Brute Force to a Real Vector Database

Searching NaijaThreads' 20 FAQs by comparing a query against every single one, by hand, is instant. Let's prove that — and then prove why NaijaThreads can't rely on this forever if the business grows.

In [ ]:
def time_brute_force_search(num_documents, vector_dim=25):
    rng = np.random.default_rng(seed=42)
    documents = rng.normal(size=(num_documents, vector_dim))
    query = rng.normal(size=vector_dim)

    start_time = time.time()
    scores = [cosine_similarity(query, documents[i]) for i in range(num_documents)]
    scores.sort(reverse=True)
    return time.time() - start_time


for size in [20, 1_000, 10_000, 100_000]:
    elapsed = time_brute_force_search(size)
    print(f"{size:>8,} documents  ->  {elapsed:.4f} seconds")

20 FAQs today — instant. But if NaijaThreads grows into a marketplace with 100,000 product listings and customer messages, this same approach slows down noticeably. This is exactly why vector databases exist — so let's give NaijaThreads one, the same way we did in Topic 4.

In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name="naijathreads_faqs",
    metadata={"hnsw:space": "cosine"},  # cosine, not the database default — see Topic 4
)

collection.add(
    embeddings=[vector.tolist() for vector in faq_embeddings],
    documents=[faq["text"] for faq in faqs],
    metadatas=[{"category": faq["category"]} for faq in faqs],
    ids=[faq["id"] for faq in faqs],
)

print("FAQs stored in ChromaDB:", collection.count())

Let's run one quick query through the database to prove the pipeline works, before we build the full customer-facing experience in Part 4.

In [ ]:
test_query_vector = sentence_vector("how can i pay", word_model)

test_results = collection.query(query_embeddings=[test_query_vector.tolist()], n_results=3)

for text, distance in zip(test_results["documents"][0], test_results["distances"][0]):
    print(f"  distance={distance:.3f}  -  {text}")

---
## Part 4 — Assembling the Full Semantic Search Engine

Now for the real test: a customer question that doesn't use NaijaThreads' exact wording at all — and a side-by-side comparison against the naive keyword search NaijaThreads' website currently uses.

In [ ]:
def keyword_search(query, documents, top_n=5):
    query_words = set(w for w in query.split() if w not in STOPWORDS)
    scored = []
    for doc in documents:
        doc_words = set(w for w in doc["text"].split() if w not in STOPWORDS)
        overlap_count = len(query_words & doc_words)
        scored.append((overlap_count, doc))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:top_n]


customer_question = "can i pay with my phone"
question_vector = sentence_vector(customer_question, word_model)

semantic_results = collection.query(query_embeddings=[question_vector.tolist()], n_results=5)
keyword_results = keyword_search(customer_question, faqs)

print(f'Customer question: "{customer_question}"\n')

print("NaijaThreads' CURRENT search (keyword):")
for overlap, faq in keyword_results:
    print(f"  overlap={overlap}  [{faq['category']:<9}]  {faq['text']}")

print()
print("NEW semantic search:")
for text, metadata, distance in zip(
    semantic_results["documents"][0], semantic_results["metadatas"][0], semantic_results["distances"][0]
):
    print(f"  distance={distance:.3f}  [{metadata['category']:<9}]  {text}")

**This is the whole story of the week, in one result.** NaijaThreads' current keyword search returns nothing useful — every FAQ ties at zero overlap. The semantic search engine we just built, using nothing but a small trained embedding model and a local vector database, correctly surfaces the payment FAQ despite sharing no words with the question at all.

This is a complete, working pipeline: **documents → chunks (not needed here, our FAQs are already short) → embeddings → vector store → query → ranked results** — exactly the five stages from Topic 5, Slide 4.

---
## The Story, End to End

| Stage | What We Did | Topic |
|-------|-------------|-------|
| 1. Words → Vectors | Trained Word2Vec on NaijaThreads' FAQs + background text | Topics 1–2 |
| 2. Measuring Similarity | Implemented cosine similarity and Euclidean distance by hand | Topic 3 |
| 3. Brute Force → Vector DB | Timed brute-force search at scale, then stored vectors in ChromaDB | Topics 3–4 |
| 4. Full Search Engine | Compared semantic search against NaijaThreads' current keyword search | Topic 5 |

NaijaThreads now has a working semantic search engine, built entirely from techniques we implemented ourselves this week — no black boxes.

**Where this goes next (Week 11):** right now, a customer still has to read the matched FAQ text themselves. Next week, we hand these same retrieved results to an LLM, and NaijaThreads' assistant will generate a direct, conversational answer instead — the retrieval half of Retrieval-Augmented Generation (RAG), built this week, put to work.